# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```

**What this notebook does**
1. Loads the environment variables (OpenAI / Tavily keys).
2. Creates a *persistent* ChromaDB store through a reusable `PersistentVectorStoreManager`.
3. Converts every game JSON file into an embedded document (name, platform, year, description, genre and publisher are all searchable).
4. Demonstrates semantic search, metadata filtering and persistence.

### Setup

In [9]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [10]:
import os
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")  # silence Chroma telemetry warnings

import json
from typing import Any, Dict, List, Optional

import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv, find_dotenv

from lib.documents import Document, Corpus
from lib.vector_db import VectorStore, VectorStoreManager

In [11]:
# Load keys from config.env (course convention) or .env, searching parent folders too.
for env_file in ("config.env", ".env"):
    env_path = find_dotenv(env_file, usecwd=True)
    if env_path:
        load_dotenv(env_path, override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
CHROMA_OPENAI_API_KEY = os.getenv("CHROMA_OPENAI_API_KEY", OPENAI_API_KEY)
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://openai.vocareum.com/v1")
os.environ["OPENAI_BASE_URL"] = OPENAI_BASE_URL  # picked up by the OpenAI client used in lib/llm.py

assert OPENAI_API_KEY, "OPENAI_API_KEY is not set"
assert CHROMA_OPENAI_API_KEY, "CHROMA_OPENAI_API_KEY is not set"
assert TAVILY_API_KEY, "TAVILY_API_KEY is not set"

print("Environment variables loaded successfully.")
print("OpenAI base URL:", OPENAI_BASE_URL)

Environment variables loaded successfully.
OpenAI base URL: https://openai.vocareum.com/v1


### Reusable vector store manager

`lib/vector_db.py` already provides `VectorStoreManager`, but it is in-memory only. We extend it (without modifying the library) so the same class can be reused in Part 02 and for long-term memory.

In [12]:
class GameVectorStore(VectorStore):
    """lib.vector_db.VectorStore plus the helpers UdaPlay needs: count / upsert / search."""

    def count(self) -> int:
        return self._collection.count()

    def upsert(self, documents: List[Document]) -> None:
        """Insert or update documents by id, so re-running the notebook never duplicates data."""
        data = Corpus(documents).to_dict()
        self._collection.upsert(
            ids=data["ids"], documents=data["contents"], metadatas=data["metadatas"]
        )

    def search(self, query: str, n_results: int = 5,
               where: Optional[Dict[str, Any]] = None) -> List[Dict[str, Any]]:
        """Semantic search returning plain dicts: game metadata + Chroma distance (lower = closer)."""
        result = self.query(query_texts=[query], n_results=n_results, where=where)
        return [
            {**metadata, "distance": round(float(distance), 4)}
            for metadata, distance in zip(result["metadatas"][0], result["distances"][0])
        ]


class PersistentVectorStoreManager(VectorStoreManager):
    """Reusable, *persistent* vector store manager.

    lib.vector_db.VectorStoreManager uses chromadb.Client() (in-memory, lost when the kernel
    stops) and the default embedding model. This subclass keeps the same interface
    (get_store / get_or_create_store / delete_store) but writes to disk and lets us choose
    the embedding model + base URL. Part 01 and Part 02 must use the same embedding model.
    """

    def __init__(self, openai_api_key: str, persist_path: str = "chromadb",
                 embedding_model: str = "text-embedding-3-small", api_base: Optional[str] = None):
        # super().__init__ is intentionally not called: it would create an in-memory client.
        self.chroma_client = chromadb.PersistentClient(path=persist_path)
        self.embedding_function = embedding_functions.OpenAIEmbeddingFunction(
            api_key=openai_api_key, model_name=embedding_model, api_base=api_base
        )

    def get_store(self, name: str) -> Optional[GameVectorStore]:
        try:
            collection = self.chroma_client.get_collection(
                name, embedding_function=self.embedding_function
            )
            return GameVectorStore(collection)
        except Exception:
            return None

    def get_or_create_store(self, store_name: str) -> GameVectorStore:
        collection = self.chroma_client.get_or_create_collection(
            name=store_name, embedding_function=self.embedding_function
        )
        return GameVectorStore(collection)

### VectorDB Instance and Collection

In [13]:
PERSIST_PATH = "chromadb"
COLLECTION_NAME = "udaplay"

manager = PersistentVectorStoreManager(
    openai_api_key=CHROMA_OPENAI_API_KEY,
    persist_path=PERSIST_PATH,
    embedding_model="text-embedding-3-small",
    api_base=OPENAI_BASE_URL,
)
store = manager.get_or_create_store(COLLECTION_NAME)

print(f"Persistent ChromaDB ready at ./{PERSIST_PATH}")
print(f"Collection '{COLLECTION_NAME}' ready. Existing documents: {store.count()}")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Persistent ChromaDB ready at ./chromadb
Collection 'udaplay' ready. Existing documents: 15


### Add documents

Genre and Publisher are included in the indexed text (and kept in the metadata), so questions such as *"Who published Halo Infinite?"* can be answered by semantic search. Documents are added with `upsert`, so re-running this cell never creates duplicates.

In [14]:
def build_game_document(game: dict, doc_id: str) -> Document:
    content = (
        f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']} "
        f"Genre: {game['Genre']}. Publisher: {game['Publisher']}."
    )
    return Document(id=doc_id, content=content, metadata=game)  # file name (e.g. 001) is the ID


def load_games(data_dir: str = "games") -> List[Document]:
    documents = []
    for file_name in sorted(os.listdir(data_dir)):
        if not file_name.endswith(".json"):
            continue
        with open(os.path.join(data_dir, file_name), "r", encoding="utf-8") as f:
            game = json.load(f)
        documents.append(build_game_document(game, os.path.splitext(file_name)[0]))
    return documents


game_documents = load_games("games")
print(f"Loaded {len(game_documents)} game files. Example document:\n  {game_documents[0].content}")

store.upsert(game_documents)
assert store.count() == len(game_documents), "Collection size does not match the number of game files"
print(f"\nIndexed {store.count()} game documents in the '{COLLECTION_NAME}' collection.")

Loaded 15 game files. Example document:
  [PlayStation 1] Gran Turismo (1997) - A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre. Genre: Racing. Publisher: Sony Computer Entertainment.

Indexed 15 game documents in the 'udaplay' collection.


### Semantic search demo

In [15]:
demo_queries = [
    "Which games are available on PlayStation?",
    "Who published Halo Infinite?",
    "A game about building and surviving in a blocky open world",
    "Which Nintendo role-playing games exist?",
]

for query in demo_queries:
    print(f"\nQuery: {query}")
    for rank, hit in enumerate(store.search(query, n_results=3), start=1):
        print(f"  {rank}. {hit['Name']} [{hit['Platform']}, {hit['YearOfRelease']}] "
              f"- {hit['Genre']} / {hit['Publisher']}  (distance={hit['distance']})")

# Semantic search combined with a metadata filter
print("\nQuery with metadata filter (Genre == 'Racing'): fast cars")
for hit in store.search("fast cars", n_results=3, where={"Genre": "Racing"}):
    print(f"  - {hit['Name']} [{hit['Platform']}] (distance={hit['distance']})")


Query: Which games are available on PlayStation?
  1. Gran Turismo 5 [PlayStation 3, 2010] - Racing / Sony Computer Entertainment  (distance=1.0426)
  2. Marvel's Spider-Man [PlayStation 4, 2018] - Action-adventure / Sony Interactive Entertainment  (distance=1.1214)
  3. Gran Turismo [PlayStation 1, 1997] - Racing / Sony Computer Entertainment  (distance=1.1477)

Query: Who published Halo Infinite?
  1. Halo Infinite [Xbox Series X|S, 2021] - First-person shooter / Xbox Game Studios  (distance=0.7221)
  2. Minecraft [Xbox One, 2014] - Sandbox, Survival / Mojang Studios  (distance=1.2902)
  3. Marvel's Spider-Man 2 [PlayStation 5, 2023] - Action-adventure / Sony Interactive Entertainment  (distance=1.3898)

Query: A game about building and surviving in a blocky open world
  1. Minecraft [Xbox One, 2014] - Sandbox, Survival / Mojang Studios  (distance=1.0113)
  2. Grand Theft Auto: San Andreas [PlayStation 2, 2004] - Action-adventure / Rockstar Games  (distance=1.3264)
  3. Marvel's Spi

### Verify persistence

A brand-new manager instance (as Part 02 will create) must see the same data on disk.

In [16]:
reopened = PersistentVectorStoreManager(
    openai_api_key=CHROMA_OPENAI_API_KEY, persist_path=PERSIST_PATH, api_base=OPENAI_BASE_URL
).get_store(COLLECTION_NAME)

assert reopened is not None and reopened.count() == len(game_documents)
print(f"Persistence verified: reopened '{COLLECTION_NAME}' contains {reopened.count()} documents.")
print("Top hit for 'When was Pokémon Gold and Silver released?':",
      reopened.search("When was Pokémon Gold and Silver released?", n_results=1)[0]["Name"])

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Persistence verified: reopened 'udaplay' contains 15 documents.
Top hit for 'When was Pokémon Gold and Silver released?': Pokémon Gold and Silver
